# SpAM Simulation Evaluation (Read-Only)

This notebook reads the results of an **already-completed** simulation run - it performs no
simulation, no MDS sweep, and needs no R/rpy2. It only loads the four small files a finished
run writes (`out/coverage.csv`, `out/stability.csv`, `out/embedding_stability.csv`,
`mds_store/meta.csv`) via `SpAM_Simulations/eval_helpers.py`, and builds every figure from
them. To generate a new run, use `evaluation.ipynb` (task-v0.1 simulation) or
`evaluation_task_v2_3.ipynb` (task-v2.3, per-subject trial design) instead.

Works unmodified for either simulation type: a task-v0.1 run has 5 levers
(`num_subjects`, `trials_per_subject`, `images_per_trial`, `subjects_noise_scale`,
`subjects_noise_df`); a task-v2.3 run adds a 6th, `frac_images_repeated`. Figures below
auto-detect which one a given run is and degrade gracefully (dropping the lever from
trace names/facets) wherever the extra lever is absent.

In [1]:
import plotly.io as pio

from SpAM_Simulations import eval_helpers as eh

pio.renderers.default = "browser"

## Load Run
Set `RUN_RESULTS_DIR` to a folder name under `SpAM_Simulations/sim_results/` (e.g. `"task-v2.3"`).
`eh.load_run` resolves it relative to `eval_helpers.py`'s own directory (not the notebook
kernel's cwd) and raises `FileNotFoundError` naming every missing file if the run directory or
any of the four expected files is absent.

In [2]:
RUN_RESULTS_DIR = "sim_results/task-v2.4"
run = eh.load_run(RUN_RESULTS_DIR)
print(f"task_version={run.task_version}, {len(run.mds_meta)} MDS rows, {len(run.coverage)} coverage rows")

task_version=2.4, 4320 MDS rows, 480 coverage rows


### Levers in this run

In [3]:
eh.lever_summary_table(run.levers).show()

## Coverage
Two coverage metrics, read directly from `out/coverage.csv` (no recomputation):
- **% Images Seen** (`img_coverage`): percentage of images observed at least once.
- **% Pairs Seen** (`pair_coverage`): percentage of image-pairs observed at least once.

Layout: rows = the two metrics above; columns = `trials_per_subject`; x-axis = `num_subjects`;
traces = `images_per_trial` and `frac_images_repeated` (the latter dropped automatically on a
task-v0.1 run, where it doesn't exist).

`metrics.coverage()` computes both quantities purely from `num_obs` (which images/pairs were
ever *drawn*), never from the noisy `distances` values - so both metrics are mathematically
independent of `subjects_noise_scale`/`subjects_noise_df`. They still vary stochastically
across `rep`, since which images land in which trial is itself a random draw - hence we
aggregate with mean +/- SEM across `rep` (and across the noise levers, which contribute no
additional variance here) before plotting.

In [4]:
GROUP_COLS = [c for c in ["num_subjects", "trials_per_subject", "images_per_trial", "frac_images_repeated", "frac_trials_repeated"]
              if c in run.coverage.columns]
coverage_summary = (
    run.coverage.groupby(GROUP_COLS)
    .agg(img_coverage_mean=("img_coverage", "mean"), img_coverage_sem=("img_coverage", "sem"),
         pair_coverage_mean=("pair_coverage", "mean"), pair_coverage_sem=("pair_coverage", "sem"))
    .reset_index()
)
eh.faceted_metric_figure(
    coverage_summary, x="num_subjects",
    metrics=[("img_coverage_mean", "img_coverage_sem", "% Images Seen"),
             ("pair_coverage_mean", "pair_coverage_sem", "% Pairs Seen")],
    col_by="trials_per_subject", trace_by=["images_per_trial", "frac_images_repeated", "frac_trials_repeated"],
    title="Coverage by Experimental Configuration", x_title="Number of Subjects",
).show()

## Connectivity (P[Connected])
`num_connected_components == 1` checks whether the observed image-pair graph is a single
connected component - a strict prerequisite for MDS. Same independence property as coverage
above: connectivity depends only on which pairs were drawn (`num_obs`), not on the noisy
distance values, so it too is independent of `subjects_noise_scale`/`subjects_noise_df` (but
still varies across `rep`, since the draws themselves are random).

Layout: y = % of reps with exactly 1 connected component; x = `num_subjects`; columns =
`trials_per_subject`; rows = `images_per_trial`; traces = `frac_images_repeated` (dropped on a
task-v0.1 run).

In [5]:
conn = run.coverage.copy()
conn["is_connected"] = conn["num_connected_components"] == 1
GROUP_COLS = [c for c in ["num_subjects", "trials_per_subject", "images_per_trial", "frac_images_repeated", "frac_trials_repeated"]
              if c in conn.columns]
connectivity_summary = (
    conn.groupby(GROUP_COLS)
    .agg(p_connected_mean=("is_connected", "mean"), p_connected_sem=("is_connected", "sem"))
    .reset_index()
)
eh.faceted_lever_figure(
    connectivity_summary, x="num_subjects", y="p_connected_mean", y_sem="p_connected_sem",
    row_by="images_per_trial", col_by="trials_per_subject", trace_by=["frac_images_repeated", "frac_trials_repeated"],
    title="P[Connected] by Experimental Configuration", x_title="Number of Subjects",
    y_title="% of reps connected",
).show()

## Test-Retest Reliability (task-v2.4)
For task-v2.4 runs (which sweep `frac_trials_repeated`), each subject who received whole-trial
repeats yields a test-retest reliability: the mean Spearman correlation between the original and
repeat presentations of their repeated trials (`mean_test_retest` in `out/coverage.csv`). The
figure shows how it tracks `subjects_noise_scale`, one trace per `frac_trials_repeated` (the
all-NaN `frac_trials_repeated = 0` slice is omitted). Skipped automatically on task-v0.1/v2.3
runs, which don't carry this metric.


In [6]:
if "mean_test_retest" in run.coverage.columns:
    eh.test_retest_figure(run.coverage).show()
else:
    print("No test-retest data in this run (task-v2.4 only).")


## Pre-MDS Stability
Spearman rank correlation between the mean observed distances of different repetitions of the
same configuration (`out/stability.csv`) - the data-reliability ceiling before MDS is even run.

Layout: y = mean Spearman correlation; x = `num_subjects`; columns = `trials_per_subject`;
rows = `images_per_trial`; traces = `subjects_noise_scale`, `subjects_noise_df`,
`frac_images_repeated` (any constant or missing feature is dropped from the trace name and
captioned instead).

In [7]:
GROUP_COLS = [c for c in eh.LEVER_COLUMNS if c in run.stability.columns]
stability_summary = (
    run.stability.dropna(subset=["spearman"])
    .groupby(GROUP_COLS)
    .agg(spearman_mean=("spearman", "mean"), spearman_sem=("spearman", "sem"))
    .reset_index()
)
eh.faceted_lever_figure(
    stability_summary, x="num_subjects", y="spearman_mean", y_sem="spearman_sem",
    row_by="images_per_trial", col_by="trials_per_subject",
    trace_by=["subjects_noise_scale", "subjects_noise_df", "frac_images_repeated", "frac_trials_repeated"],
    title="Pre-MDS Stability by Experimental Configuration", x_title="Number of Subjects",
    y_title="Spearman R",
).show()

## MDS Scree Plots
MDS stress vs. target dimensionality, read directly from `mds_store/meta.csv` (successful
runs only - `status` in `{success, max_iters}`).

Layout: y = mean stress; x = `ndim`; columns = `trials_per_subject`; rows = `images_per_trial`;
traces = `num_subjects`, `subjects_noise_scale`, `subjects_noise_df` (a trace-feature with only
one value in this run is removed from the trace name and reported in the figure caption
instead). If `frac_images_repeated` exists, one separate figure is produced per value
(labeled in its title), rather than folding it into the trace name.

In [8]:
success = run.mds_meta[run.mds_meta["status"].isin(["success", "max_iters"])]
GROUP_COLS = [c for c in ["num_subjects", "trials_per_subject", "images_per_trial",
                          "subjects_noise_scale", "subjects_noise_df", "ndim"]
              if c in success.columns]

# Slice by whichever repetition lever(s) vary in this run (frac_images_repeated on a task-v2.3
# run, frac_trials_repeated on a task-v2.4 run), so panels never average across them.
for caption, df in eh.repeat_lever_slices(success):
    stress_summary = df.groupby(GROUP_COLS)["stress"].agg(stress_mean="mean", stress_sem="sem").reset_index()
    title = "MDS Stress by Dimension" + (f" ({caption})" if caption else "")
    eh.faceted_lever_figure(
        stress_summary, x="ndim", y="stress_mean", y_sem="stress_sem",
        row_by="images_per_trial", col_by="trials_per_subject",
        trace_by=["num_subjects", "subjects_noise_scale", "subjects_noise_df"],
        title=title, x_title="Target Dimensionality", y_title="Stress",
    ).show()


## Post-MDS (Embedding) Stability
Mean Spearman agreement of reconstructed MDS distances (`confdist`) across repetitions,
already computed by `pipeline.compute_embedding_stability` and stored in
`out/embedding_stability.csv` - no recomputation, no touching `confdists.f32`.

Same layout and single-value/`frac_images_repeated`-splitting rules as the scree plots above:
y = `mean_spearman`; x = `ndim`; columns = `trials_per_subject`; rows = `images_per_trial`;
traces = `num_subjects`, `subjects_noise_scale`, `subjects_noise_df`.

In [9]:
# Same per-repetition-lever slicing as the scree plots above.
for caption, df in eh.repeat_lever_slices(run.embedding_stability):
    title = "Embedding Stability by Dimension" + (f" ({caption})" if caption else "")
    eh.faceted_lever_figure(
        df, x="ndim", y="mean_spearman", y_sem="sem_spearman",
        row_by="images_per_trial", col_by="trials_per_subject",
        trace_by=["num_subjects", "subjects_noise_scale", "subjects_noise_df"],
        title=title, x_title="Target Dimensionality", y_title="Spearman R",
    ).show()


## Drill-Down: Focus Configuration
The overview figures above average over many configurations at once. Here we fix every lever
except `num_subjects` to a single value (the focus configuration) and re-examine coverage, MDS
convergence, and stability for just that slice - mirroring `evaluation.ipynb`'s "Final
Configuration Evaluation" section, but reading from the same pre-loaded `run` instead of
recomputing anything.

Available combinations of every lever except `num_subjects` (from `mds_store/meta.csv`):

In [10]:
SECONDARY_LEVERS = [l for l in eh.LEVER_COLUMNS if l != "num_subjects" and l in run.mds_meta.columns]
eh.available_configs(run.mds_meta, SECONDARY_LEVERS)

,trials_per_subject,images_per_trial,subjects_noise_scale,subjects_noise_df,frac_images_repeated,frac_trials_repeated
0,10,20,0.5,1,0.0,0.0
1,10,20,0.5,1,0.0,0.1
2,10,20,0.5,1,0.0,0.2
3,10,20,0.5,1,0.0,0.3
4,10,20,0.8,1,0.0,0.0
5,10,20,0.8,1,0.0,0.1
6,10,20,0.8,1,0.0,0.2
7,10,20,0.8,1,0.0,0.3
8,15,20,0.5,1,0.0,0.0
9,15,20,0.5,1,0.0,0.1


In [12]:
# Pick one configuration from the table above. Any key not relevant to this run (e.g.
# frac_images_repeated on a task-v0.1 run, or frac_trials_repeated on a task-v2.3 run) is
# dropped automatically below. Pin BOTH repetition levers so a task-v2.4 drill-down isn't
# pooled across frac_trials_repeated.
FOCUS_CONFIG = {
    "trials_per_subject": 20, "images_per_trial": 20,
    "subjects_noise_scale": 0.8, "subjects_noise_df": 1,
    "frac_images_repeated": 0.0, "frac_trials_repeated": 0.1,
}
FOCUS_CONFIG = {k: v for k, v in FOCUS_CONFIG.items() if k in run.mds_meta.columns}
assert not eh.filter_to_config(run.mds_meta, FOCUS_CONFIG).empty, (
    f"No rows match {FOCUS_CONFIG} - check the available-configs table above"
)
FOCUS_CONFIG

{'trials_per_subject': 20,
 'images_per_trial': 20,
 'subjects_noise_scale': 0.8,
 'subjects_noise_df': 1,
 'frac_images_repeated': 0.0,
 'frac_trials_repeated': 0.1}

#### Coverage

In [13]:
cov = eh.filter_to_config(run.coverage, FOCUS_CONFIG)
cov_summary = (
    cov.groupby("num_subjects")
    .agg(img_coverage_mean=("img_coverage", "mean"), img_coverage_sem=("img_coverage", "sem"),
         pair_coverage_mean=("pair_coverage", "mean"), pair_coverage_sem=("pair_coverage", "sem"))
    .reset_index()
)
eh.faceted_metric_figure(
    cov_summary, x="num_subjects",
    metrics=[("img_coverage_mean", "img_coverage_sem", "% Images Seen"),
             ("pair_coverage_mean", "pair_coverage_sem", "% Pairs Seen")],
    title="Coverage - Focus Configuration", x_title="Number of Subjects",
).show()

#### Convergence

In [14]:
meta = eh.filter_to_config(run.mds_meta, FOCUS_CONFIG)
eh.convergence_bar_figure(meta).show()

#### Stability: Pre- vs. Post-MDS

In [15]:
emb = eh.filter_to_config(run.embedding_stability, FOCUS_CONFIG)
stab = eh.filter_to_config(run.stability, FOCUS_CONFIG)
eh.pre_post_mds_stability_figure(emb, stab).show()